In [ ]:
import itertools
import os
import tempfile
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

from recommenders.evaluation.python_evaluation import (
    map_at_k, ndcg_at_k, precision_at_k, recall_at_k,
)
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.models.ncf.ncf_singlenode import NCF as MSRecNCF
from recommenders.utils.constants import (
    DEFAULT_ITEM_COL, DEFAULT_RATING_COL,
    DEFAULT_TIMESTAMP_COL, DEFAULT_USER_COL,
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

LAYER_CONFIGS = {
    'small':  [64, 32],
    'medium': [128, 64],
    'large':  [256, 128, 64],
    'xlarge': [256, 128, 64, 32],
}
K_VALUES      = [5, 10, 20]
SEARCH_K      = 10
EVAL_USERS    = 500
SEARCH_EPOCHS = 10
FINAL_EPOCHS  = 20

print(f'Device: {DEVICE}')
print('Imports OK')

In [ ]:
df = pd.read_csv('../data/children_products/clildren_product_cleaned.csv')

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

MIN_INTERACTIONS = 3
user_counts = df_filtered.groupby('Телефон_new').size()
item_counts = df_filtered.groupby('ID_SKU').size()
df_filtered = df_filtered[
    df_filtered['Телефон_new'].isin(user_counts[user_counts >= MIN_INTERACTIONS].index) &
    df_filtered['ID_SKU'].isin(item_counts[item_counts >= MIN_INTERACTIONS].index)
]

print(f'Пользователей: {df_filtered["Телефон_new"].nunique():,}')
print(f'Товаров:       {df_filtered["ID_SKU"].nunique():,}')
print(f'Взаимодействий:{len(df_filtered):,}')

In [ ]:
interactions = (
    df_filtered
    .groupby(['Телефон_new', 'ID_SKU'])
    .agg(timestamp=('Дата', 'max'), mean_date=('Дата', 'mean'))
    .reset_index()
)
interactions[DEFAULT_RATING_COL] = 1.0
interactions.rename(columns={
    'Телефон_new': DEFAULT_USER_COL,
    'ID_SKU':      DEFAULT_ITEM_COL,
    'timestamp':   DEFAULT_TIMESTAMP_COL,
}, inplace=True)

interactions = interactions.sort_values(DEFAULT_TIMESTAMP_COL)
split_ts = interactions[DEFAULT_TIMESTAMP_COL].quantile(0.7)
split_date = pd.Timestamp(split_ts)
print(f'Дата разделения: {split_date}')

train_df = interactions[interactions[DEFAULT_TIMESTAMP_COL] <  split_ts].copy()
test_df  = interactions[interactions[DEFAULT_TIMESTAMP_COL] >= split_ts].copy()

train_users = set(train_df[DEFAULT_USER_COL].unique())
test_df = test_df[test_df[DEFAULT_USER_COL].isin(train_users)]

print(f'Train: {len(train_df):,} пар, {train_df[DEFAULT_USER_COL].nunique():,} users')
print(f'Test:  {len(test_df):,} пар,  {test_df[DEFAULT_USER_COL].nunique():,} users')

In [ ]:
# --- Кодировка ID для MS Recommenders NCF ---
user2id = {u: i for i, u in enumerate(interactions[DEFAULT_USER_COL].unique())}
item2id = {it: i for i, it in enumerate(interactions[DEFAULT_ITEM_COL].unique())}

def encode_ms(df, u2id=None, i2id=None):
    if u2id is None: u2id = user2id
    if i2id is None: i2id = item2id
    d = df.copy()
    d[DEFAULT_USER_COL] = d[DEFAULT_USER_COL].map(u2id)
    d[DEFAULT_ITEM_COL] = d[DEFAULT_ITEM_COL].map(i2id)
    return d.dropna(subset=[DEFAULT_USER_COL, DEFAULT_ITEM_COL])

train_enc = encode_ms(train_df).sort_values(DEFAULT_USER_COL)
test_enc  = encode_ms(test_df).sort_values(DEFAULT_USER_COL)

tmp_dir    = tempfile.mkdtemp()
train_file = os.path.join(tmp_dir, 'train.csv')
test_file  = os.path.join(tmp_dir, 'test.csv')

train_enc[[DEFAULT_USER_COL, DEFAULT_ITEM_COL, DEFAULT_RATING_COL]].to_csv(train_file, index=False)
test_enc[[DEFAULT_USER_COL, DEFAULT_ITEM_COL, DEFAULT_RATING_COL]].to_csv(test_file,  index=False)

data = NCFDataset(
    train_file=train_file, test_file=test_file, seed=42, n_neg=4, n_neg_test=99,
    col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
)
print(f'NCFDataset: {data.n_users:,} users, {data.n_items:,} items')

In [ ]:
# --- User features из showcase CSV ---
showcase = pd.read_csv('../data/children_products/clildren_product_showcase_remove_corr.csv', low_memory=False)
phone_col = 'phone' if 'phone' in showcase.columns else 'Телефон_new'
showcase = showcase.rename(columns={phone_col: 'phone'})

# Гео и метод доставки (бинарные)
geo_cols = [c for c in showcase.columns if c.startswith('Гео_')]
delivery_cols = [c for c in showcase.columns if c.startswith('МетодДоставки_Групп_')]

# K-Means кластер → one-hot
n_clusters = int(showcase['kmeans_cluster'].nunique())
cluster_ohe = pd.get_dummies(showcase['kmeans_cluster'].astype(int), prefix='cluster').astype(float)

# Числовые статы
stat_cols = ['СредняяСуммаЗаказов', 'КоличествоЗаказов', 'ДоляОтменненыхЗаказов', 'КоличествоУникальныхТоваров']
stat_cols = [c for c in stat_cols if c in showcase.columns]

user_feat_df = pd.concat([
    showcase[['phone']],
    cluster_ohe.values if isinstance(cluster_ohe, pd.DataFrame) else cluster_ohe,
    showcase[geo_cols],
    showcase[delivery_cols],
    showcase[stat_cols],
], axis=1)
# Переименуем, чтобы не было конфликтов
cluster_col_names = [f'cluster_{i}' for i in range(n_clusters)]
user_feat_df.columns = ['phone'] + cluster_col_names + geo_cols + delivery_cols + stat_cols
user_feat_df = user_feat_df.set_index('phone')

print(f'User feature matrix: {user_feat_df.shape}')
print(f'  cluster:  {cluster_col_names}  ({len(cluster_col_names)} dims)')
print(f'  geo:      {geo_cols}  ({len(geo_cols)} dims)')
print(f'  delivery: {delivery_cols}  ({len(delivery_cols)} dims)')
print(f'  stats:    {stat_cols}  ({len(stat_cols)} dims)')

In [ ]:
# --- Item features: Группа2 (наиболее частая категория на item_id) ---
item_cat = (
    df_filtered.groupby('ID_SKU')['Группа2']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
cat_encoder = LabelEncoder()
item_cat['cat_id'] = cat_encoder.fit_transform(item_cat['Группа2'])
n_categories = len(cat_encoder.classes_)

item_feat_df = pd.get_dummies(item_cat.set_index('ID_SKU')['cat_id'], prefix='cat').astype(float)

print(f'Item feature matrix: {item_feat_df.shape}  ({n_categories} categories)')
print(f'Категории Группа2: {list(cat_encoder.classes_)}')

In [ ]:
rng = np.random.default_rng(42)
test_users_eval = [
    u for u in test_enc[DEFAULT_USER_COL].unique()
    if u in set(user2id.values())
]
eval_sample = rng.choice(
    test_users_eval,
    size=min(EVAL_USERS, len(test_users_eval)),
    replace=False
).tolist()


def quick_eval_ms(model, eval_users, test_enc, k=SEARCH_K):
    known_users = set(model.user2id.keys())
    known_items = sorted(model.item2id.keys())
    users = [u for u in eval_users if u in known_users]
    preds = []
    for u in users:
        scores = model.predict([u] * len(known_items), known_items, is_list=True)
        for item, score in zip(known_items, scores):
            preds.append({DEFAULT_USER_COL: u, DEFAULT_ITEM_COL: item, DEFAULT_RATING_COL: score})
    pred_df = pd.DataFrame(preds)
    test_sub = test_enc[
        test_enc[DEFAULT_USER_COL].isin(set(users)) &
        test_enc[DEFAULT_ITEM_COL].isin(set(known_items))
    ]
    return ndcg_at_k(
        test_sub, pred_df, k=k,
        col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
        col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL,
    )


def full_eval_ms(model, test_enc, k_values=K_VALUES):
    known_users = set(model.user2id.keys())
    known_items = sorted(model.item2id.keys())
    test_users  = [u for u in test_enc[DEFAULT_USER_COL].unique() if u in known_users]
    preds = []
    for u in test_users:
        scores = model.predict([u] * len(known_items), known_items, is_list=True)
        for item, score in zip(known_items, scores):
            preds.append({DEFAULT_USER_COL: u, DEFAULT_ITEM_COL: item, DEFAULT_RATING_COL: score})
    pred_df = pd.DataFrame(preds)
    test_sub = test_enc[
        test_enc[DEFAULT_USER_COL].isin(set(test_users)) &
        test_enc[DEFAULT_ITEM_COL].isin(set(known_items))
    ]
    results = {}
    for k in k_values:
        results[k] = {
            'precision': precision_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
            'recall':    recall_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
            'map':       map_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
            'ndcg':      ndcg_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
        }
    return results


print(f'Quick-eval sample: {len(eval_sample)} users')

In [ ]:
print('Обучение baseline NCF (NeuMF, 20 эпох)...')
t0 = time.time()
baseline_model = MSRecNCF(
    n_users=data.n_users, n_items=data.n_items,
    model_type='NeuMF', n_factors=64,
    layer_sizes=[256, 128, 64], n_epochs=FINAL_EPOCHS,
    batch_size=1024, learning_rate=1e-3, verbose=2, seed=42,
)
baseline_model.fit(data)
print(f'Готово за {time.time() - t0:.0f}s')

In [ ]:
baseline_results = full_eval_ms(baseline_model, test_enc)
print('=== Baseline MS-NCF ===')
for k in K_VALUES:
    r = baseline_results[k]
    print(f'  K={k}: P={r["precision"]:.4f}  R={r["recall"]:.4f}  MAP={r["map"]:.4f}  NDCG={r["ndcg"]:.4f}')

### 3. Поиск гиперпараметров (Optuna)

In [ ]:
def make_ncf_objective(train_file, test_file, eval_users, test_enc, n_epochs=SEARCH_EPOCHS):
    def objective(trial):
        n_factors = trial.suggest_int('n_factors', 16, 128)
        lr        = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        layer_key = trial.suggest_categorical('layers', list(LAYER_CONFIGS.keys()))
        n_neg     = trial.suggest_int('n_neg', 2, 8)

        data_t = NCFDataset(
            train_file=train_file, test_file=test_file, seed=42,
            n_neg=n_neg, n_neg_test=99,
            col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
        )
        m = MSRecNCF(
            n_users=data_t.n_users, n_items=data_t.n_items,
            model_type='NeuMF', n_factors=n_factors,
            layer_sizes=LAYER_CONFIGS[layer_key], n_epochs=n_epochs,
            batch_size=1024, learning_rate=lr, verbose=0, seed=42,
        )
        m.fit(data_t)
        return quick_eval_ms(m, eval_users, test_enc, k=SEARCH_K)
    return objective


N_TRIALS = 40
print(f'Запуск Optuna: {N_TRIALS} trials, {SEARCH_EPOCHS} эпох каждый...')
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    make_ncf_objective(train_file, test_file, eval_sample, test_enc),
    n_trials=N_TRIALS, show_progress_bar=True,
)
print(f'\nЛучший NDCG@{SEARCH_K}: {study.best_value:.4f}')
print(f'Лучшие параметры: {study.best_params}')

In [ ]:
trials_df = study.trials_dataframe()
top10 = trials_df.sort_values('value', ascending=False).head(10)
display(top10[['number', 'value', 'params_n_factors', 'params_lr', 'params_layers', 'params_n_neg']]
        .rename(columns={'value': f'NDCG@{SEARCH_K}'}))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trials_df['number'], trials_df['value'], alpha=0.5, linewidth=1)
ax.plot(trials_df['number'],
        trials_df['value'].cummax(), linewidth=2, color='red', label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel(f'NDCG@{SEARCH_K}')
ax.set_title('Optuna: прогресс поиска гиперпараметров')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
bp = study.best_params
BEST_N_FACTORS = bp['n_factors']
BEST_LR        = bp['lr']
BEST_LAYERS    = LAYER_CONFIGS[bp['layers']]
BEST_N_NEG     = bp['n_neg']
print(f'Лучшая конфигурация: n_factors={BEST_N_FACTORS}, lr={BEST_LR:.2e}, layers={BEST_LAYERS}, n_neg={BEST_N_NEG}')

data_best = NCFDataset(
    train_file=train_file, test_file=test_file, seed=42,
    n_neg=BEST_N_NEG, n_neg_test=99,
    col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
)
t0 = time.time()
best_ms_model = MSRecNCF(
    n_users=data_best.n_users, n_items=data_best.n_items,
    model_type='NeuMF', n_factors=BEST_N_FACTORS,
    layer_sizes=BEST_LAYERS, n_epochs=FINAL_EPOCHS,
    batch_size=1024, learning_rate=BEST_LR, verbose=2, seed=42,
)
best_ms_model.fit(data_best)
print(f'Готово за {time.time() - t0:.0f}s')

best_ms_results = full_eval_ms(best_ms_model, test_enc)
print(f'\n{"K":>4}  {"Метрика":>12}  {"Baseline":>9}  {"BestHP":>9}  {"Δ":>8}')
print('-' * 52)
for k in K_VALUES:
    for m in ['precision', 'recall', 'map', 'ndcg']:
        base = baseline_results[k][m]
        best = best_ms_results[k][m]
        d = best - base
        print(f'{k:>4}  {m+"@"+str(k):>12}  {base:>9.4f}  {best:>9.4f}  {"+" if d>=0 else ""}{d:>7.4f}')

In [ ]:
train_with_time = encode_ms(train_df).copy()

days_before_split = (
    split_date - train_df['mean_date']
).dt.total_seconds().values / 86400
days_before_split = np.clip(days_before_split, 0, None)

DECAY_LAMBDAS = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
decay_scores  = []

for lam in DECAY_LAMBDAS:
    t0 = time.time()
    w = np.exp(-lam * days_before_split).clip(1e-6)
    train_d = train_with_time[[DEFAULT_USER_COL, DEFAULT_ITEM_COL]].copy()
    train_d[DEFAULT_RATING_COL] = w

    decay_file = os.path.join(tmp_dir, f'train_decay_{lam}.csv')
    train_d.to_csv(decay_file, index=False)

    data_d = NCFDataset(
        train_file=decay_file, test_file=test_file, seed=42,
        n_neg=BEST_N_NEG, n_neg_test=99,
        col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
    )
    m_d = MSRecNCF(
        n_users=data_d.n_users, n_items=data_d.n_items,
        model_type='NeuMF', n_factors=BEST_N_FACTORS,
        layer_sizes=BEST_LAYERS, n_epochs=SEARCH_EPOCHS,
        batch_size=1024, learning_rate=BEST_LR, verbose=0, seed=42,
    )
    m_d.fit(data_d)
    score = quick_eval_ms(m_d, eval_sample, test_enc, k=SEARCH_K)
    decay_scores.append(score)
    print(f'  λ={lam:.3f}  NDCG@{SEARCH_K}={score:.4f}  ({time.time()-t0:.0f}s)')

BEST_LAMBDA = DECAY_LAMBDAS[int(np.argmax(decay_scores))]
print(f'\nЛучшая λ = {BEST_LAMBDA}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(DECAY_LAMBDAS, decay_scores, marker='o', linewidth=2)
ax.axvline(BEST_LAMBDA, color='red', linestyle='--', alpha=0.6, label=f'λ*={BEST_LAMBDA}')
ax.set_xscale('log')
ax.set_xlabel('λ (log scale)')
ax.set_ylabel(f'NDCG@{SEARCH_K} (quick eval, {EVAL_USERS} users)')
ax.set_title('Влияние временного затухания на качество NCF')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
w_best = np.exp(-BEST_LAMBDA * days_before_split).clip(1e-6)
train_best_d = train_with_time[[DEFAULT_USER_COL, DEFAULT_ITEM_COL]].copy()
train_best_d[DEFAULT_RATING_COL] = w_best
decay_best_file = os.path.join(tmp_dir, 'train_decay_best.csv')
train_best_d.to_csv(decay_best_file, index=False)

data_decay_best = NCFDataset(
    train_file=decay_best_file, test_file=test_file, seed=42,
    n_neg=BEST_N_NEG, n_neg_test=99,
    col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
)
t0 = time.time()
decay_model = MSRecNCF(
    n_users=data_decay_best.n_users, n_items=data_decay_best.n_items,
    model_type='NeuMF', n_factors=BEST_N_FACTORS,
    layer_sizes=BEST_LAYERS, n_epochs=FINAL_EPOCHS,
    batch_size=1024, learning_rate=BEST_LR, verbose=2, seed=42,
)
decay_model.fit(data_decay_best)
print(f'Готово за {time.time() - t0:.0f}s')

decay_results = full_eval_ms(decay_model, test_enc)

---
## Блок B — FeatureAwareNCF (PyTorch)
### 5. Модель и ablation study по фичам

Расширяем NeuMF из `ncf.ipynb`: user/item фичи проецируются линейным слоем и прибавляются к соответствующим эмбеддингам (residual addition).

In [ ]:
# --- Подготовка данных для PyTorch модели (та же логика что в ncf.ipynb) ---

user_encoder_pt = LabelEncoder()
item_encoder_pt = LabelEncoder()

all_users_pt = df_filtered['Телефон_new'].unique()
all_items_pt = df_filtered['ID_SKU'].unique()
user_encoder_pt.fit(all_users_pt)
item_encoder_pt.fit(all_items_pt)

n_users_pt = len(user_encoder_pt.classes_)
n_items_pt = len(item_encoder_pt.classes_)

# Temporal split для PyTorch ноутбука: сохраняем split_date из выше
df_interactions = df_filtered.sort_values('Дата').copy()
split_date_pt = df_interactions['Дата'].quantile(0.8)  # 80/20 как в ncf.ipynb
train_pt = df_interactions[df_interactions['Дата'] < split_date_pt].copy()
test_pt  = df_interactions[df_interactions['Дата'] >= split_date_pt].copy()

train_pt['user_id'] = user_encoder_pt.transform(train_pt['Телефон_new'])
train_pt['item_id'] = item_encoder_pt.transform(train_pt['ID_SKU'])
test_pt['user_id']  = user_encoder_pt.transform(test_pt['Телефон_new'])
test_pt['item_id']  = item_encoder_pt.transform(test_pt['ID_SKU'])

train_interactions_pt = train_pt.groupby(['user_id', 'item_id']).size().reset_index(name='count')
test_interactions_pt  = test_pt.groupby(['user_id', 'item_id']).size().reset_index(name='count')
train_users_pt = set(train_interactions_pt['user_id'].unique())

print(f'PyTorch split — Train: {len(train_interactions_pt):,}, Test: {len(test_interactions_pt):,}')
print(f'Train users: {len(train_users_pt):,}')

In [ ]:
def generate_negatives(interactions_df, n_items, num_neg=4):
    user_items = interactions_df.groupby('user_id')['item_id'].apply(set).to_dict()
    neg_samples = []
    rng_neg = np.random.default_rng(42)
    for uid in user_items:
        pos = user_items[uid]
        for _ in range(len(pos) * num_neg):
            for _a in range(50):
                cand = int(rng_neg.integers(0, n_items))
                if cand not in pos:
                    neg_samples.append({'user_id': uid, 'item_id': cand, 'label': 0})
                    break
    return pd.DataFrame(neg_samples)


train_pos_pt = train_interactions_pt[['user_id', 'item_id']].copy()
train_pos_pt['label'] = 1

# Hard negatives (отмены + возвраты)
df_hard = df[(df['Статус'].isin(['Отменен', 'Возврат'])) | (df['Отменено'] == 'Да')].copy()
df_hard['Дата'] = pd.to_datetime(df_hard['Дата'], errors='coerce')
df_hard = df_hard.dropna(subset=['Телефон_new', 'ID_SKU'])
df_hard = df_hard[
    df_hard['Телефон_new'].isin(set(user_encoder_pt.classes_)) &
    df_hard['ID_SKU'].isin(set(item_encoder_pt.classes_))
].copy()
df_hard['user_id'] = user_encoder_pt.transform(df_hard['Телефон_new'])
df_hard['item_id'] = item_encoder_pt.transform(df_hard['ID_SKU'])

pos_pairs = set(zip(train_interactions_pt['user_id'], train_interactions_pt['item_id']))
hard_neg_pt = df_hard[['user_id', 'item_id']].drop_duplicates()
hard_neg_pt = hard_neg_pt[~hard_neg_pt.apply(lambda r: (r['user_id'], r['item_id']) in pos_pairs, axis=1)]
hard_neg_pt = hard_neg_pt.copy()
hard_neg_pt['label'] = 0

easy_neg_pt = generate_negatives(train_interactions_pt, n_items_pt, num_neg=4)

train_samples_pt = pd.concat([train_pos_pt, hard_neg_pt, easy_neg_pt], ignore_index=True)
train_samples_pt = train_samples_pt.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Train samples: {len(train_samples_pt):,}  (pos={len(train_pos_pt):,}, hard_neg={len(hard_neg_pt):,}, easy={len(easy_neg_pt):,})')

In [ ]:
# Матрица user фичей (индекс = user_id из PyTorch encoder)
uid_to_phone = dict(zip(
    user_encoder_pt.transform(user_encoder_pt.classes_),
    user_encoder_pt.classes_
))

cluster_cols = [c for c in user_feat_df.columns if c.startswith('cluster_')]
geo_feat_cols = [c for c in user_feat_df.columns if c.startswith('Гео_')]
del_feat_cols = [c for c in user_feat_df.columns if c.startswith('МетодДоставки_Групп_')]
num_feat_cols = stat_cols

# Нормализуем числовые фичи по train пользователям
train_phones = [uid_to_phone[uid] for uid in train_interactions_pt['user_id'].unique() if uid in uid_to_phone]
scaler = StandardScaler()
train_showcase = user_feat_df.loc[user_feat_df.index.isin(train_phones), num_feat_cols].fillna(0)
scaler.fit(train_showcase)

def build_user_feat_matrix(col_groups):
    """Строит матрицу user фичей формы [n_users, feat_dim]."""
    feat_cols = [c for g in col_groups for c in g]
    if not feat_cols:
        return None, 0
    mat = np.zeros((n_users_pt, len(feat_cols)), dtype=np.float32)
    for uid, phone in uid_to_phone.items():
        if phone in user_feat_df.index:
            row = user_feat_df.loc[phone, feat_cols].fillna(0).values
            # нормализуем числовые
            num_mask = [c in num_feat_cols for c in feat_cols]
            if any(num_mask):
                num_vals = row[[i for i, m in enumerate(num_mask) if m]].reshape(1, -1)
                num_idx  = [i for i, m in enumerate(num_mask) if m]
                norm_col_names = [feat_cols[i] for i in num_idx]
                norm_vals = scaler.transform(
                    pd.DataFrame([num_vals[0]], columns=norm_col_names)
                )[0]
                for i, v in zip(num_idx, norm_vals):
                    row[i] = v
            mat[uid] = row.astype(np.float32)
    return torch.tensor(mat, dtype=torch.float32), len(feat_cols)


# Item feature matrix формы [n_items, n_categories]
iid_to_sku = dict(zip(
    item_encoder_pt.transform(item_encoder_pt.classes_),
    item_encoder_pt.classes_
))
item_feat_mat = np.zeros((n_items_pt, n_categories), dtype=np.float32)
for iid, sku in iid_to_sku.items():
    if sku in item_feat_df.index:
        item_feat_mat[iid] = item_feat_df.loc[sku].values.astype(np.float32)
item_feat_tensor = torch.tensor(item_feat_mat, dtype=torch.float32)

print('Матрицы фичей построены')
print(f'  Item feature matrix: {item_feat_tensor.shape}')

In [ ]:
class FeatureAwareDataset(Dataset):
    def __init__(self, user_ids, item_ids, labels, user_feat_mat, item_feat_mat):
        self.user_ids = torch.LongTensor(user_ids)
        self.item_ids = torch.LongTensor(item_ids)
        self.labels   = torch.FloatTensor(labels)
        self.user_feat_mat = user_feat_mat   # может быть None
        self.item_feat_mat = item_feat_mat   # может быть None

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        uid = self.user_ids[idx]
        iid = self.item_ids[idx]
        u_feat = self.user_feat_mat[uid] if self.user_feat_mat is not None else torch.zeros(1)
        i_feat = self.item_feat_mat[iid] if self.item_feat_mat is not None else torch.zeros(1)
        return uid, iid, self.labels[idx], u_feat, i_feat


class FeatureAwareNCF(nn.Module):
    """
    NeuMF с опциональными user/item feature projections.
    Фичи проецируются через Linear(feat_dim → emb_dim//2) и
    добавляются к эмбеддингам в обеих ветках (GMF и MLP).
    """
    def __init__(self, num_users, num_items,
                 user_feat_dim=0, item_feat_dim=0,
                 emb_dim=32, hidden_dim=32, dropout=0.3):
        super().__init__()
        proj_dim = emb_dim // 2

        self.user_embedding_gmf = nn.Embedding(num_users, emb_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, emb_dim)
        self.user_embedding_mlp = nn.Embedding(num_users, emb_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, emb_dim)

        self.user_feat_proj = nn.Linear(user_feat_dim, proj_dim) if user_feat_dim > 0 else None
        self.item_feat_proj = nn.Linear(item_feat_dim, proj_dim) if item_feat_dim > 0 else None

        # Если есть фичи — GMF/MLP input вырастает на proj_dim
        u_extra = proj_dim if user_feat_dim > 0 else 0
        i_extra = proj_dim if item_feat_dim > 0 else 0
        gmf_out_dim = emb_dim + u_extra + i_extra  # для поэлементного произведения нужна одинаковая размерность
        # Упрощаем: проекцию добавляем через pad до emb_dim при необходимости
        # Вместо этого используем конкатенацию в GMF и финальный linear
        self.gmf_out_dim = emb_dim  # поэлементное произведение только на ID-эмбеддингах

        mlp_in = emb_dim * 2 + u_extra + i_extra
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2), nn.BatchNorm1d(hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.ReLU(),
        )

        fc_in = emb_dim + hidden_dim // 2
        self.fc_out = nn.Linear(fc_in, 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [self.user_embedding_gmf, self.item_embedding_gmf,
                    self.user_embedding_mlp, self.item_embedding_mlp]:
            nn.init.normal_(emb.weight, std=0.01)
        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.constant_(self.fc_out.bias, 0)
        if self.user_feat_proj is not None:
            nn.init.xavier_uniform_(self.user_feat_proj.weight)
            nn.init.constant_(self.user_feat_proj.bias, 0)
        if self.item_feat_proj is not None:
            nn.init.xavier_uniform_(self.item_feat_proj.weight)
            nn.init.constant_(self.item_feat_proj.bias, 0)

    def forward(self, user_ids, item_ids, user_feats=None, item_feats=None):
        u_gmf = self.user_embedding_gmf(user_ids)
        v_gmf = self.item_embedding_gmf(item_ids)
        gmf_out = u_gmf * v_gmf

        u_mlp = self.user_embedding_mlp(user_ids)
        v_mlp = self.item_embedding_mlp(item_ids)

        mlp_parts = [u_mlp, v_mlp]
        if self.user_feat_proj is not None and user_feats is not None:
            mlp_parts.append(self.user_feat_proj(user_feats))
        if self.item_feat_proj is not None and item_feats is not None:
            mlp_parts.append(self.item_feat_proj(item_feats))

        mlp_out = self.mlp(torch.cat(mlp_parts, dim=-1))
        x = torch.cat([gmf_out, mlp_out], dim=-1)
        return torch.sigmoid(self.fc_out(x).squeeze(-1))


print('FeatureAwareNCF и FeatureAwareDataset определены')

In [ ]:
def precision_at_k_pt(rec, rel, k):
    return len(set(rec[:k]) & set(rel)) / k if k > 0 else 0.0

def recall_at_k_pt(rec, rel, k):
    return len(set(rec[:k]) & set(rel)) / len(rel) if rel else 0.0

def map_at_k_pt(rec, rel, k):
    rel_set = set(rel)
    hits = score = 0.0
    for i, item in enumerate(rec[:k]):
        if item in rel_set:
            hits += 1; score += hits / (i + 1)
    return score / min(len(rel_set), k) if rel_set else 0.0

def ndcg_at_k_pt(rec, rel, k):
    rel_set = set(rel)
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(rec[:k]) if item in rel_set)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel_set), k)))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_model_pt(model, train_int, test_int, n_items, user_feat_mat, item_feat_mat,
                      k_values=(5, 10, 20), device=DEVICE, n_eval_users=None):
    model.eval()
    test_user_items = test_int.groupby('user_id')['item_id'].apply(list).to_dict()
    train_users_set = set(train_int['user_id'].unique())
    eval_users = [u for u in test_user_items if u in train_users_set]
    if n_eval_users:
        eval_users = eval_users[:n_eval_users]

    train_hist = train_int.groupby('user_id')['item_id'].apply(set).to_dict()
    metrics = {k: {'precision': [], 'recall': [], 'map': [], 'ndcg': []} for k in k_values}

    items_t = torch.arange(n_items, device=device, dtype=torch.long)
    with torch.no_grad():
        for uid in eval_users:
            users_t = torch.full_like(items_t, uid)
            u_f = user_feat_mat[uid].unsqueeze(0).expand(n_items, -1).to(device) if user_feat_mat is not None else None
            i_f = item_feat_mat.to(device) if item_feat_mat is not None else None
            scores = model(users_t, items_t, u_f, i_f).cpu().numpy()

            bought = train_hist.get(uid, set())
            for i in bought:
                scores[i] = -1e9
            top_items = np.argsort(-scores)[:max(k_values)].tolist()
            relevant = test_user_items[uid]
            for k in k_values:
                metrics[k]['precision'].append(precision_at_k_pt(top_items, relevant, k))
                metrics[k]['recall'].append(recall_at_k_pt(top_items, relevant, k))
                metrics[k]['map'].append(map_at_k_pt(top_items, relevant, k))
                metrics[k]['ndcg'].append(ndcg_at_k_pt(top_items, relevant, k))

    return {k: {m: float(np.mean(v)) for m, v in mv.items()} for k, mv in metrics.items()}


print('evaluate_model_pt определена')

In [ ]:
def train_feature_ncf(user_feat_mat, item_feat_mat, user_feat_dim, item_feat_dim,
                      n_epochs=15, batch_size=256, lr=0.0005, emb_dim=32, hidden_dim=32,
                      device=DEVICE, verbose=True):
    dataset = FeatureAwareDataset(
        train_samples_pt['user_id'].values,
        train_samples_pt['item_id'].values,
        train_samples_pt['label'].values,
        user_feat_mat, item_feat_mat,
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = FeatureAwareNCF(
        n_users_pt, n_items_pt,
        user_feat_dim=user_feat_dim,
        item_feat_dim=item_feat_dim,
        emb_dim=emb_dim, hidden_dim=hidden_dim,
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    losses = []

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        it = tqdm(loader, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False) if verbose else loader
        for uid, iid, lbl, u_f, i_f in it:
            uid, iid, lbl = uid.to(device), iid.to(device), lbl.to(device)
            u_f = u_f.to(device) if user_feat_dim > 0 else None
            i_f = i_f.to(device) if item_feat_dim > 0 else None
            pred = model(uid, iid, u_f, i_f)
            loss = criterion(pred, lbl)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader)
        losses.append(avg)
        if verbose:
            print(f'  Epoch {epoch+1}: loss={avg:.4f}')
    return model, losses


# Определяем 6 вариантов ablation
feat_mat_cluster, dim_cluster       = build_user_feat_matrix([cluster_cols])
feat_mat_clust_geo, dim_clust_geo   = build_user_feat_matrix([cluster_cols, geo_feat_cols, del_feat_cols])
feat_mat_clust_stat, dim_clust_stat = build_user_feat_matrix([cluster_cols, num_feat_cols])
feat_mat_user_all, dim_user_all     = build_user_feat_matrix([cluster_cols, geo_feat_cols, del_feat_cols, num_feat_cols])

ABLATION_VARIANTS = [
    ('base',           None,               0,           None,             0),
    ('cluster',        feat_mat_cluster,   dim_cluster, None,             0),
    ('cluster_geo',    feat_mat_clust_geo, dim_clust_geo, None,           0),
    ('cluster_stats',  feat_mat_clust_stat, dim_clust_stat, None,         0),
    ('user_all',       feat_mat_user_all,  dim_user_all, None,            0),
    ('user_all_item',  feat_mat_user_all,  dim_user_all, item_feat_tensor, n_categories),
]

print('Варианты ablation:')
for name, _, udim, _, idim in ABLATION_VARIANTS:
    print(f'  {name:<18} user_feat_dim={udim:>3}  item_feat_dim={idim:>3}')

In [ ]:
ablation_results = {}

for name, u_feat_mat, u_feat_dim, i_feat_mat, i_feat_dim in ABLATION_VARIANTS:
    print(f'\n>>> Вариант: {name}  (user_feat_dim={u_feat_dim}, item_feat_dim={i_feat_dim})')
    t0 = time.time()
    m, losses = train_feature_ncf(
        u_feat_mat, i_feat_mat, u_feat_dim, i_feat_dim,
        n_epochs=15, verbose=True,
    )
    res = evaluate_model_pt(
        m, train_interactions_pt, test_interactions_pt,
        n_items_pt, u_feat_mat, i_feat_mat,
        k_values=K_VALUES, device=DEVICE,
    )
    ablation_results[name] = res
    print(f'  Время: {time.time()-t0:.0f}s')
    for k in K_VALUES:
        r = res[k]
        print(f'  K={k}: P={r["precision"]:.4f}  R={r["recall"]:.4f}  MAP={r["map"]:.4f}  NDCG={r["ndcg"]:.4f}')

In [ ]:
base_res = ablation_results['base']
print(f'{"Вариант":>18}  {"K":>4}  {"NDCG":>8}  {"MAP":>8}  {"vs base NDCG":>13}')
print('-' * 65)
for name in [v[0] for v in ABLATION_VARIANTS]:
    res = ablation_results[name]
    for k in K_VALUES:
        ndcg = res[k]['ndcg']
        mmap = res[k]['map']
        d = ndcg - base_res[k]['ndcg']
        print(f'{name:>18}  {k:>4}  {ndcg:>8.4f}  {mmap:>8.4f}  {"+" if d>=0 else ""}{d:>11.4f}')

# Bar chart NDCG@10 по вариантам
variant_names = [v[0] for v in ABLATION_VARIANTS]
ndcg10 = [ablation_results[n][10]['ndcg'] for n in variant_names]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(variant_names, ndcg10, color=sns.color_palette('tab10', len(variant_names)))
ax.axhline(ndcg10[0], color='gray', linestyle='--', alpha=0.5, label='base')
for bar, v in zip(bars, ndcg10):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('NDCG@10')
ax.set_title('Ablation study: влияние фичей пользователя/товара на NCF')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Тепловая карта: variant × metric@K
metric_names = ['precision', 'recall', 'map', 'ndcg']
hmap_data = []
hmap_cols = []
for k in K_VALUES:
    for m in metric_names:
        hmap_cols.append(f'{m}@{k}')

for name in variant_names:
    row = []
    for k in K_VALUES:
        for m in metric_names:
            row.append(ablation_results[name][k][m])
    hmap_data.append(row)

hmap_df = pd.DataFrame(hmap_data, index=variant_names, columns=hmap_cols)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(hmap_df, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Ablation: все метрики × все K')
plt.tight_layout()
plt.show()

In [ ]:
ms_variants = {
    'baseline_ms':   baseline_results,
    'best_hparams_ms': best_ms_results,
    'time_decay_ms': decay_results,
}
pt_variants = {f'{n}_pt': ablation_results[n] for n in variant_names}

all_variants = {**ms_variants, **pt_variants}

# Используем baseline_ms как точку отсчёта (близко к ncf_ms_recommenders.ipynb)
ref = baseline_results

print(f'{"Вариант":>22}  {"NDCG@10":>9}  {"MAP@10":>9}  {"ΔNDCG@10":>10}')
print('-' * 60)
for name, res in all_variants.items():
    ndcg = res[10]['ndcg']
    mmap = res[10]['map']
    d = ndcg - ref[10]['ndcg']
    print(f'{name:>22}  {ndcg:>9.4f}  {mmap:>9.4f}  {"+" if d>=0 else ""}{d:>9.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = sns.color_palette('tab20', len(all_variants))

for ax, metric in zip(axes, ['ndcg', 'map']):
    for (name, res), color in zip(all_variants.items(), colors):
        y = [res[k][metric] for k in K_VALUES]
        ls = '-' if '_ms' in name else '--'
        ax.plot(K_VALUES, y, marker='o', linewidth=1.5, linestyle=ls, label=name, color=color)
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K: все варианты')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle('NCF Experiments — Итоговое сравнение  (—: MS Recommenders,  --: PyTorch)', fontsize=12)
plt.tight_layout()
plt.show()